# **Chess Dataset Apriori Analysis**

This notebook is designed to demonstrate the principles of **Apriori Analysis** using the **Chess dataset**. The goal is to uncover frequent patterns in chess games and generate association rules that provide actionable insights.

---

## **Objective**

The main objective is to analyze chess games to identify:
- Common game outcomes based on player ratings and opening strategies.
- Relationships between moves, openings, and game results.
- Patterns and associations that occur frequently in chess games.

---

## **Dataset Overview**

### **File Structure**
The dataset contains the following columns:
- **Categorical Attributes**:
  - `victory_status`: Outcome of the game (e.g., win, draw).
  - `winner`: The winning player (e.g., white, black).
  - `opening_name`: Name of the chess opening.
  - `opening_eco`: ECO code of the chess opening.

- **Numerical Attributes**:
  - `turns`: Total number of moves in the game.
  - `white_rating`: Rating of the white player.
  - `black_rating`: Rating of the black player.

Each row represents a single chess game.

---

## **Analysis Workflow**

### **1. Dataset Preprocessing**
- **Goal**: Convert raw chess data into a transactional format suitable for Apriori analysis.
- **Steps**:
  1. Extract key categorical attributes (`victory_status`, `winner`, `opening_name`, etc.).
  2. Categorize numerical attributes (`turns`, `white_rating`, `black_rating`) into ranges (e.g., `High`, `Low`) for pattern mining.
  3. Represent each game as a transaction containing meaningful items (e.g., `Winner_White`, `Opening_Sicilian`).

### **2. Frequent Itemset Mining**
- **Goal**: Identify itemsets that appear frequently across transactions.
- **Technique**:
  - Use the Apriori algorithm to generate frequent itemsets based on a minimum support threshold.
  - For example:
    - `{Winner_White}`: Indicates games where white is the winner.
    - `{Opening_Sicilian, VictoryStatus_Win}`: Indicates frequent patterns involving the Sicilian opening and a win.

### **3. Association Rules Generation**
- **Goal**: Derive actionable rules from the frequent itemsets.
- **Metrics**:
  - **Support**: Proportion of games containing the itemset.
  - **Confidence**: Likelihood of the consequent given the antecedent.
  - **Lift**: Strength of the association compared to random chance.

For example:
- Rule: `{Winner_White} => {VictoryStatus_Win}`
  - Support: 35%
  - Confidence: 78%
  - Lift: 1.2

### **4. Insights and Applications**
- **Insights**:
  - Understand the impact of openings on game outcomes.
  - Discover relationships between player ratings and game strategies.
  - Identify frequent patterns that lead to wins or draws.

- **Applications**:
  - Help players improve strategies based on data-driven patterns.
  - Provide insights for chess analysts and coaches.

---

## **Parameters for Analysis**

### **Adjustable Thresholds**
- **Minimum Support**:
  - Controls the frequency threshold for itemsets.
  - Example: `min_support = 0.3` considers itemsets appearing in at least 30% of games.

- **Minimum Confidence**:
  - Filters rules based on the strength of the association.
  - Example: `min_confidence = 0.7` selects rules where the consequent is likely to occur 70% of the time.

---

## **Expected Outcomes**

### **Frequent Itemsets**
- Identification of common patterns such as:
  - `{VictoryStatus_Win, Opening_Sicilian}`: Frequently occurring wins with the Sicilian opening.
  - `{Turns_High, Winner_White}`: Games with high turns where white wins.

### **Association Rules**
- Actionable rules linking game attributes:
  - Rule: `{Opening_RuyLopez} => {Winner_White}`
    - Support: 25%
    - Confidence: 80%
    - Lift: 1.5

---

## **Requirements**
- **Python 3.6+**
- Libraries:
  - `pandas`: For data preprocessing.
  - `mlxtend`: For Apriori and association rules analysis.

---

## **How to Use**
1. **Prepare the Dataset**:
   - Ensure the chess dataset is available in CSV format with the specified file structure.

2. **Follow the Workflow**:
   - Preprocess the dataset into transactions.
   - Apply the Apriori algorithm to mine frequent itemsets.
   - Generate and interpret association rules.

3. **Explore Insights**:
   - Adjust thresholds (`min_support`, `min_confidence`) to refine the analysis.
   - Use the insights to improve chess strategies and understand game dynamics.

---

## **Conclusion**

This analysis provides a comprehensive understanding of frequent patterns and associations in chess games. By leveraging data from real games, it highlights actionable insights for players, coaches, and analysts. The Apriori algorithm's application demonstrates the power of frequent itemset mining in uncovering hidden patterns within structured datasets.

In [82]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Preprocessing Chess Dataset
def preprocess_chess_data(filename, nrows=None):
    data = pd.read_csv(filename, nrows=nrows)

    # Transform each row into a transaction
    transactions = []
    for _, row in data.iterrows():
        transaction = []

        # Add categorical attributes
        transaction.append(f"VictoryStatus_{row['victory_status']}")
        transaction.append(f"Winner_{row['winner']}")
        transaction.append(f"Opening_{row['opening_name']}")
        transaction.append(f"OpeningECO_{row['opening_eco']}")

        # Categorize numerical attributes
        transaction.append(f"Turns_{'High' if row['turns'] > 40 else 'Low'}")
        transaction.append(f"WhiteRating_{'High' if row['white_rating'] > 1600 else 'Low'}")
        transaction.append(f"BlackRating_{'High' if row['black_rating'] > 1600 else 'Low'}")

        transactions.append(transaction)
    return transactions

# Load and preprocess the dataset
filename = "Data/chess.csv"  # Replace with your path
transactions = preprocess_chess_data(filename)

# Convert transactions to one-hot encoded DataFrame
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_)

# Apply Apriori
min_support = 0.3
frequent_itemsets = apriori(df, min_support=min_support, use_colnames=True)

# Calculate num_itemsets
num_itemsets = frequent_itemsets['itemsets'].apply(len).max()

# Generate association rules
rules = association_rules(frequent_itemsets, num_itemsets=num_itemsets, metric="confidence", min_threshold=0.7)

# Display frequent itemsets
print("Frequent Itemsets:")
print(frequent_itemsets)

# Display association rules
print("\nAssociation Rules:")
print(rules)

Frequent Itemsets:
     support                              itemsets
0   0.449347                    (BlackRating_High)
1   0.550653                     (BlackRating_Low)
2   0.703709                          (Turns_High)
3   0.315336                  (VictoryStatus_mate)
4   0.555738                (VictoryStatus_resign)
5   0.458072                    (WhiteRating_High)
6   0.541928                     (WhiteRating_Low)
7   0.454033                        (Winner_black)
8   0.498604                        (Winner_white)
9   0.343703        (BlackRating_High, Turns_High)
10  0.340712  (BlackRating_High, WhiteRating_High)
11  0.360006         (BlackRating_Low, Turns_High)
12  0.433293    (BlackRating_Low, WhiteRating_Low)
13  0.306312       (Winner_white, BlackRating_Low)
14  0.357114    (VictoryStatus_resign, Turns_High)
15  0.346046        (WhiteRating_High, Turns_High)
16  0.357663         (WhiteRating_Low, Turns_High)
17  0.324060            (Turns_High, Winner_black)
18  0.341460